In [2]:
import pandas as pd

df = pd.read_excel('data/Online Retail.xlsx')

df = df.dropna()
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

print(df.shape)

(397884, 8)


In [3]:
df['amount'] = df['Quantity'] * df['UnitPrice']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['hour'] = df['InvoiceDate'].dt.hour
df['day_of_week'] = df['InvoiceDate'].dt.dayofweek

In [11]:
df['amount_log'] = np.log1p(df['amount'])
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [12]:
import numpy as np

df['is_fraud'] = 0

df.loc[(df['amount'] > 12000) & (df['hour'] < 6), 'is_fraud'] = 1
df.loc[(df['amount'] > 8000) & (df['day_of_week'] == 6), 'is_fraud'] = 1

random_idx = np.random.choice(df.index, size=int(0.02 * len(df)), replace=False)
df.loc[random_idx, 'is_fraud'] = 1

df['is_fraud'].value_counts()

is_fraud
0    389927
1      7957
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

features = ['amount', 'amount_log', 'hour', 'day_of_week', 'is_weekend']

X = df[features]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
rf = RandomForestClassifier(class_weight='balanced')
xgb = XGBClassifier(scale_pos_weight=20, eval_metric='logloss')

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, name):
    y_pred = model.predict(X_test)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))
    print("Recall:", recall_score(y_test, y_pred, zero_division=0))
    print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))

evaluate(lr, "Logistic Regression")
evaluate(rf, "Random Forest")
evaluate(xgb, "XGBoost")


Logistic Regression
Accuracy: 0.4546916822700026
Precision: 0.019936149199568202
Recall: 0.5455688246385921
F1 Score: 0.03846665189452692

Random Forest
Accuracy: 0.7577063724442993
Precision: 0.021374458874458876
Recall: 0.2482715273412948
F1 Score: 0.03936027103781575

XGBoost
Accuracy: 0.9706573507420486
Precision: 0.015625
Recall: 0.00754242614707731
F1 Score: 0.010173802458668928


In [16]:
import pandas as pd

results = []

def evaluate_table(model, name):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append([name, acc, prec, rec, f1])

evaluate_table(lr, "Logistic Regression")
evaluate_table(rf, "Random Forest")
evaluate_table(xgb, "XGBoost")

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"])

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.454692,0.019936,0.545569,0.038467
1,Random Forest,0.757706,0.021374,0.248272,0.039360
2,XGBoost,0.970657,0.015625,0.007542,0.010174
